# Missing Values {#sec-missing-values}

## Introduction

In this chapter, we'll look at the tools and tricks for dealing with missing values.  We'll start by discussing some general tools for working with missing values recorded as `null` values. We'll then explore the idea of implicitly missing values, values are that are simply absent from your data, and show some tools you can use to make them explicit.
We'll finish off with a related discussion of empty groups, caused by categories that don't appear in the data.

In [ ]:
# remove cell
import matplotlib.pyplot as plt
import matplotlib_inline.backend_inline

# Plot settings
plt.style.use("https://github.com/aeturrell/python4DS/raw/main/plot_style.txt")
matplotlib_inline.backend_inline.set_matplotlib_formats("svg")

### Prerequisites

This chapter will use the **polars** data analysis package.

## Explicit Missing Values

To begin, let's explore a few handy tools for creating or eliminating missing explicit values, i.e. cells where you see an `null` value.

### Types of Missing Values

In polars, missing values are represented by `null`:

In [ ]:
import numpy as np
import polars as pl

df = pl.DataFrame({"numbers": [5.0, 27.3, None, -16.0]})
df

Polars also handles Python's built-in None values seamlessly:

In [ ]:
numbers = pl.DataFrame({"numbers": [None, 27.3, np.nan, -16.0, None]})
numbers

For string columns, missing values are also represented by `null`:

In [ ]:
fruits = pl.DataFrame({"fruit": ["orange", None, "apple", None, "banana", None]})
fruits

Both types of missing value can be found using the `.is_null()` method, which returns a new column of boolean values that are `True` if the value is missing:

In [ ]:
fruits.select(pl.col("fruit").is_null())

As a convenience, there is also an .is_not_null() method:

In [ ]:
fruits.select(pl.col("fruit").is_not_null())

### Dealing with Explicit Missing Values

There are various options for dealing with missing values. The `fill_null()` method achieves this. Let's take a look at it with some test data:

In [ ]:
null_df = pl.DataFrame(
    {
        "A": [None, 3, 5, None],
        "B": [2, 4, None, 3],
        "C": [None, None, None, None],
        "D": [0, 1, None, 4],
    }
)
null_df

First, we can just fill any missing values with a single fixed value:

In [ ]:
null_df.fill_null(0)

This can be done on a by-column basis by passing a dictionary:

In [ ]:
null_df.with_columns(
    [
        pl.col("A").fill_null(0),
        pl.col("B").fill_null(1),
        pl.col("C").fill_null(2),
        pl.col("D").fill_null(3),
    ]
)

We can also propagate non-null values forward or backward:

In [ ]:
null_df.fill_null(strategy="forward")

In [ ]:
null_df.fill_null(strategy="backward")

The forward fill and backward fill options are particularly useful for time series—but be careful using them if you're doing a forecasting exercise!

Another feature of all of these functions is that you can limit the number of nulls that get replaced using the `limit=` keyword argument.

In [ ]:
# Forward fill with limit of 1
null_df.fill_null(strategy="forward", limit=2)

Of course, another option might be just to filter out the missing values altogether. The `.drop_nulls()` method removes rows with any missing values:

In [ ]:
null_df.drop_nulls()

You can also drop rows where all values are null using a custom filter:

In [ ]:
null_exp_df = pl.DataFrame(
    {
        "A": [None, 3, 5, None, None],
        "B": [2, 4, None, 3, None],
        "C": [None, None, None, None, None],
        "D": [0, 1, None, 4, None],
    }
)

null_exp_df

In [ ]:
null_exp_df.filter(~pl.all_horizontal(pl.all().is_null()))

Notice that the only the last all null column was removed.

There is a `thresh` keyword (for threshold)—this allows you to keep only rows or columns containing at most a certain number of missing observations.

Another way to filter out nulls is to use the same filtering methods you would use normally, via boolean columns, in combination with the `.is_not_null()` method. In the below example, we see the rows for which column A is not null:

In [ ]:
null_df.filter(pl.col("A").is_not_null())

### Adding NA values

Sometimes you'll hit the opposite problem where some concrete value actually represents a missing value. This typically arises in data generated by older software that doesn't have a proper way to represent missing values, so it must instead use some special value like 99 or -999.

If possible, handle this when reading in the data, for example, by using the `null_values=` keyword argument when calling `pl.read_csv()`. If you discover the problem later, or your data source doesn't provide a way to handle it on reading the file, you can use a range of options to replace the given data:

In [ ]:
stata_df = pl.DataFrame(
    {
        "A": [3, -7, -99],
        "B": [4, 4, 6],
        "C": [5, -99, 5],
    }
)
stata_df

The easiest option is probably the expression-level `replace()` method:

In [ ]:
stata_df.with_columns(pl.all().replace(-99, None))

Because `replace()` accepts a dictionary, it's possible to replace several values at once:

In [ ]:
stata_df.with_columns(pl.all().replace({-99: None, -7: None}))

Note that this applies to *every* column in the data frame. To apply it to just one, just select that specific column.

## Implicit Missing Values

So far we've talked about missing values that are **explicitly** missing, i.e. you can see an `NA` or similar in your data.
But missing values can also be **implicitly** missing, if an entire row of data is simply absent from the data.
Let's illustrate the difference with a simple data set that records the price of some stock each quarter:

In [ ]:
stocks = pl.DataFrame(
    {
        "year": [2020, 2020, 2020, 2020, 2021, 2021, 2021],
        "qtr": [1, 2, 3, 4, 2, 3, 4],
        "price": [1.88, 0.59, 0.35, None, 0.92, 0.17, 2.66],
    }
)
stocks

This dataset has two missing observations:

-   The `price` in the fourth quarter of 2020 is explicitly missing, because its value is `NA`.

-   The `price` for the first quarter of 2021 is implicitly missing, because it simply does not appear in the dataset.

One way to think about the difference is with this Zen-like koan:

> An explicit missing value is the presence of an absence.
>
> An implicit missing value is the absence of a presence.

Sometimes you want to make implicit missings explicit in order to have something physical to work with.
In other cases, explicit missings are forced upon you by the structure of the data and you want to get rid of them.
The following sections discuss some tools for moving between implicit and explicit missingness.

### Pivoting

You've already seen one tool that can make implicit missings explicit and vice versa: pivoting. Making data wider can make implicit missing values explicit because every combination of the rows and new columns must have some value For example, if we pivot `stocks` to put `quarter` in the columns (and make `year` the index), both missing values become explicit:

In [ ]:
stocks.pivot(index="year", on="qtr", values="price", aggregate_function="first")

By default, making data longer preserves explicit missing values.

### Missing Values in Categorical Variables

A final type of missingness is the empty group, a group that doesn't contain any observations, which can arise when working with categorical data. 

For example, imagine we have a dataset that contains some health information about people:

In [ ]:
health = pl.DataFrame(
    {
        "name": ["Ikaia", "Oletta", "Leriah", "Dashay", "Tresaun"],
        "smoker": ["no", "no", "previously", "no", "yes"],
        "age": [34, 88, 75, 47, 56],
    }
)
health = health.with_columns(
    pl.col("smoker").cast(pl.Enum(["no", "previously", "yes"]))
)

Now we drop the last row of data:

In [ ]:
health_cut = health[:-1]
health_cut

The value 'yes' for smoker now doesn't (seem to) appear anywhere in our data frame. Because we used an `Enum` type, the column still remembers that 'yes' is a valid category. We can inspect the categories of the column's datatype:

In [ ]:
health_cut["smoker"].dtype.categories

If we perform aggregation operations, Polars will only return results for groups present in the data by default. If we need to include all categories (even those with no observations), we can join the categories list back to the aggregated result:

In [ ]:
all_smokers = pl.DataFrame({"smoker": health_cut["smoker"].dtype.categories})
means = (
    health_cut.group_by("smoker")
    .agg(pl.col("age").mean())
    .with_columns(pl.col("smoker").cast(pl.Utf8))
)
all_smokers.join(means, on="smoker", how="left")

You can see here that, because we took the mean of a number that doesn't exist, we got a null in place of a real value for the yes row (but there is a 'yes' row).